# Лабораторная работа №1 — Воспроизводимый проект: данные и пайплайн

---
Цель — собрать проект, результат которого воспроизводится другим человеком на другой машине одной последовательностью команд, без переписки с автором.

Задачи:

1.	Завести репозиторий с инженерной структурой и зафиксированным окружением.
2.	Поставить данные под версионный контроль и связать версию данных с коммитом кода.
3.	Описать обработку как пайплайн с явными зависимостями, пересчитывающий только изменившееся.
4.	Показать переход между двумя версиями данных в обе стороны.
Работы ЛР1–ЛР4 наращивают один и тот же репозиторий, ломать уже сданное нельзя.


Вариант 7 — Online News Popularity (UCI), регрессия: число репостов (`shares`).



## 1. Проверка секретов

Проверяем наличие токенов для GitHub и DogsHub. Они были загружены в вкладку слева — «Секреты».

In [ ]:
import os
from google.colab import userdata

SECRETS = {}
for name in ("GITHUB_USER", "GITHUB_TOKEN", "DAGSHUB_USER", "DAGSHUB_TOKEN"):
    try:
        v = userdata.get(name)
        os.environ[name] = v
        SECRETS[name] = v
        print(f"{name}: {'OK' if v else 'ПУСТО'}")
    except Exception:
        print(f"{name}: НЕ НАЙДЕН — добавь секрет и перезапусти")

GITHUB_USER   = SECRETS["GITHUB_USER"].strip()
GITHUB_TOKEN  = SECRETS["GITHUB_TOKEN"].strip()
DAGSHUB_USER  = SECRETS["DAGSHUB_USER"].strip()
DAGSHUB_TOKEN = SECRETS["DAGSHUB_TOKEN"].strip()
REPO_NAME = "lr1newspopularity"

GITHUB_USER: OK
GITHUB_TOKEN: OK
DAGSHUB_USER: OK
DAGSHUB_TOKEN: OK


## 2. Установка uv

Через pip install -q uv ставим uv — менеджер зависимостей, который нужен, чтобы дальше сгенерировать uv.lock (детерминированный lock-файл) и развернуть окружение из него. Необходим для воспроизводимости.

In [ ]:
!pip install -q uv
print('uv установлен.')

uv установлен.


## 3. Клонирование репозитория

Подключаем текущую среду Google Colab к удалённому репозиторию на GitHub.

1. REPO_DIR = Path(f"/content/{REPO_NAME}") -- Задаем расположение репозитория. Каталог /content — стандартная рабочая область Google Colab.
2. Далее, авторизируемся, чтобы подтвердить права пользователя на выполнение клонирования. Для этого в ячейке определена функция _auth_header(). Она берёт имя пользователя и токен GitHub и объединяет их в строку. Затем формируется HTTP-заголовок авторизации, который передаётся Git-командам.
3. Также есть функция _mask(s). Она заменяет значения GitHub- и DagsHub-токенов на *** в переданной строке. Сделано это для того, чтобы токены автоматически маскировались в диагностических сообщениях.
4. Функция git_push(msg_extra="") предназначена для отправки локальных коммитов и тегов в GitHub. Она выполняет команду git push origin main --follow-tags:
* origin — имя удалённого репозитория, которое Git автоматически назначает при клонировании;
* main — ветка, в которую отправляются изменения;
* --follow-tags — дополнительно отправляет связанные с отправляемыми коммитами теги.
5. Перед клонированием проверяем, if (REPO_DIR / ".git").exists(), где наличие .git означает, что в указанной папке уже находится репозиторий. Если репозиторий уже клонирован, повторное клонирование пропускается. Если локальной копии ещё нет, запускается команда git clone. В неё передаётся HTTPS-адрес репозитория на GitHub, а также путь, куда нужно поместить скачанные файлы.
6. После получения локальной копии репозитория задаются имя и электронная почта пользователя. В качестве имени используется значение GITHUB_USER, а адрес формируется в специальном формате GitHub username@users.noreply.github.com.
Нужно чтобы Git мог корректно подписывать новые коммиты.
7. Команда os.chdir(REPO_DIR) меняет текущую рабочую директорию Python на папку репозитория. Таким образом, все относительные пути, например src/, data/, models/ и reports/, будут относиться к корню проекта.
Далее выполняется команда, git checkout -B main, которая переключает на ветку main.

In [ ]:
import os, base64, subprocess
from pathlib import Path

REPO_DIR = Path(f"/content/{REPO_NAME}")

def _auth_header():
    raw = base64.b64encode(f"{GITHUB_USER}:{GITHUB_TOKEN}".encode()).decode()
    return f"http.https://github.com/.extraheader=AUTHORIZATION: basic {raw}"

def _mask(s: str) -> str:
    return s.replace(GITHUB_TOKEN, "***").replace(DAGSHUB_TOKEN, "***")

def git_push(msg_extra=""):
    r = subprocess.run(
        ["git", "-c", "credential.helper=", "-c", _auth_header(),
         "push", "origin", "main", "--follow-tags"],
        cwd=REPO_DIR, capture_output=True, text=True,
    )
    if r.returncode == 0:
        print("git push: OK", msg_extra)
    else:
        print("git push ошибка:", _mask(r.stderr))

if (REPO_DIR / ".git").exists():
    print("Репозиторий уже клонирован — пропускаем.")
else:
    r = subprocess.run(
        ["git", "-c", _auth_header(), "clone",
         f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git", str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(_mask(r.stderr))
    print("Репозиторий клонирован.")

subprocess.run(["git", "config", "user.email", f"{GITHUB_USER}@users.noreply.github.com"], cwd=REPO_DIR)
subprocess.run(["git", "config", "user.name",  GITHUB_USER], cwd=REPO_DIR)
os.chdir(REPO_DIR)

subprocess.run(["git", "checkout", "-B", "main"], cwd=REPO_DIR, check=False)
print("Рабочая папка:", os.getcwd())
print("Ветка:", subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                               cwd=REPO_DIR, capture_output=True, text=True).stdout.strip())

Репозиторий клонирован.
Рабочая папка: /content/lr1newspopularity
Ветка: main


## 4. Структура папок

Создаем структуру каталогов проекта, настраиваем .gitignore.
1. Создаем следующие каталоги:
* src — Python-модули, отвечающие за проверку данных, подготовку выборок, формирование признаков, обучение и оценку модели.
* notebooks  — ноутбуки Jupyter, используемые для создания кода
* tests — автоматические тесты
* data/external/ — внешние исходные данные, полученные из сторонних источников.
* data/raw/  — исходные данные в первоначальном виде, используемые проектом.
* data/processed/ — данные после предварительной обработки, проверки и разделения на обучающую и тестовую выборки.
* data/features/ — подготовленные признаки и целевые переменные, которые непосредственно используются при обучении модели.
* models/ — сохранённые обученные модели.
* reports/plots/ — отчёты и графические материалы

Параметр exist_ok=True предотвращает ошибку, если каталог уже существует.

2. Затем проверяем наличие файла src/__init__.py. Если его нет, создаётся файл с коротким описанием проекта.
3. Создание и обновление файла .gitignore. Вместо полной перезаписи, сначала определяем, какие правила отсутствуют. Если каких-либо правил из списка не хватает, они добавляются в конец файла. Уже существующие строки при этом сохраняются. Если все необходимые правила присутствуют, файл не изменяется.

In [ ]:
from pathlib import Path

for folder in [
    "src", "notebooks", "tests",
    "data/external", "data/raw", "data/processed", "data/features",
    "models", "reports/plots",
]:
    Path(folder).mkdir(parents=True, exist_ok=True)

init = Path("src/__init__.py")
if not init.exists():
    init.write_text('"""Код проекта Online News Popularity."""\n', encoding="utf-8")

gitignore = Path(".gitignore")
base = gitignore.read_text(encoding="utf-8") if gitignore.exists() else ""
needed = [
    "__pycache__/",
    "*.pyc",
    ".ruff_cache/",
    ".pytest_cache/",
    ".ipynb_checkpoints/",
    "/data/external/*",
    "!/data/external/*.dvc",
    "/data/raw/*",
    "!/data/raw/*.dvc",
    "/data/processed/",
    "/data/features/",
    "/models/",
    "/reports/",
]
missing = [line for line in needed if line not in base.splitlines()]
if missing:
    gitignore.write_text(base.rstrip("\n") + "\n" + "\n".join(missing) + "\n",
                         encoding="utf-8")
    print(f"В .gitignore добавлено {len(missing)} строк.")
else:
    print(".gitignore уже полон.")

print("Структура создана.")

В .gitignore добавлено 13 строк.
Структура создана.


## 5. pyproject.toml и окружение из uv.lock

По заданию, создаем файл pyproject.toml, в котором описываются параметры Python-проекта, его зависимости и настройки инструментов разработки. Затем с помощью менеджера зависимостей формируется файл uv.lock, а необходимые библиотеки устанавливаются в текущую среду Google Colab.

1. Создание pyproject.toml. Файл записывается с помощью Path("pyproject.toml").write_text(). Для формирования содержимого используется последовательность строк, которые объединяются в единый текстовый файл.

В начале файла находится секция [project]. В ней указываются:

- name — название проекта
- version — текущая версия проекта
- description — краткое описание
- requires-python — минимальная версия Python
- dependencies — основные библиотеки, необходимые для работы проекта.

2. В секции dependencies перечислены библиотеки, необходимые для обработки данных и построения ML-пайплайна.
Стоит отметить библиотеки:
* pyyaml -- для чтения конфигурационных параметров из файла `params.yaml`.
* joblib --для сохранения обученной модели на диск и последующей загрузки при оценке качества.
* dvc[s3] — инструмент Data Version Control с поддержкой удалённого хранилища через S3-совместимый интерфейс.
* pandera -- для проверки структуры и корректности табличных данных. С его помощью можно контролировать наличие обязательных столбцов, типы значений и ограничения на данные.

3. [dependency-groups], содержащая группу dev, содержит инструменты, нужные преимущественно во время разработки и проверки проекта.
- ruff — проверка качества кода, поиск ошибок и автоматическое форматирование;
- pre-commit — запуск проверок перед созданием коммита;
- pytest — выполнение автоматических тестов.

4. В [build-system] задаем инструменты, используемые для сборки проекта:
- hatchling — система сборки Python-пакета;
- build-backend — указывает, какой механизм должен использоваться при сборке.

5. В секции [tool.ruff] задаем параметры проверки кода:
- line-length = 88 — рекомендуемая максимальная длина строки;
- target-version = "py312" — целевая версия Python для анализа кода;
- extend-exclude = ["notebooks"] — исключение ноутбуков из стандартной проверки Ruff.

В [tool.ruff.lint] перечисляются группы правил, которые будут использоваться при анализе исходного кода. Они находят синтаксические и логические проблемы, ошибки импорта, потенциально небезопасные конструкции и особенности работы с NumPy и pandas.

6. После записи pyproject.toml запускается команда uv lock --python-preference only-system, котора анализирует зависимости проекта и формирует lock-файл. Он фиксирует дерево зависимостей. В нём сохраняются выбранные версии пакетов и информация, необходимая для воспроизведения окружения.

Параметр --python-preference only-system указывает uv предпочитать системную установку Python.

Если создание lock-файла завершается ошибкой, код выводит предупреждение. При этом используются дополнительные установки библиотек через pip.

7. Если uv.lock существует, выполняется экспорт зависимостей. Таким образом создаётся файл requirements.lock.txt, содержащий список разрешённых зависимостей. Этот файл передаётся в pip, чтобы библиотеки устанавливались на основе зависимостей.


In [ ]:
from pathlib import Path
import subprocess

Path("pyproject.toml").write_text(
    '[project]\n'
    'name = "lr1newspopularity"\n'
    'version = "0.1.0"\n'
    'description = "Лабораторные работы MLOps, вариант 7 — Online News Popularity"\n'
    'requires-python = ">=3.10"\n'
    'dependencies = [\n'
    '    "pandas>=2.2,<3.0",\n'
    '    "numpy>=1.26",\n'
    '    "scikit-learn>=1.5,<1.7",\n'
    '    "pyyaml>=6.0",\n'
    '    "joblib>=1.4",\n'
    '    "dvc[s3]>=3.55,<4.0",\n'
    '    "pandera>=0.20",\n'
    ']\n'
    '\n'
    '[dependency-groups]\n'
    'dev = [\n'
    '    "ruff>=0.7",\n'
    '    "pre-commit>=4.0",\n'
    '    "pytest>=8.3",\n'
    ']\n'
    '\n'
    '[build-system]\n'
    'requires = ["hatchling"]\n'
    'build-backend = "hatchling.build"\n'
    '\n'
    '[tool.hatch.build.targets.wheel]\n'
    'packages = ["src"]\n'
    '\n'
    '[tool.ruff]\n'
    'line-length = 88\n'
    'target-version = "py312"\n'
    'extend-exclude = ["notebooks"]\n'
    '\n'
    '[tool.ruff.lint]\n'
    'select = ["E", "W", "F", "I", "B", "UP", "NPY", "PD"]\n'
    '\n'
    '[tool.pytest.ini_options]\n'
    'testpaths = ["tests"]\n',
    encoding="utf-8",
)
print("pyproject.toml записан.")

r = subprocess.run(["uv", "lock", "--python-preference", "only-system"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("uv lock предупреждение:", r.stderr[:500])
else:
    print("uv lock: OK")

if Path("uv.lock").exists():
    r2 = subprocess.run(
        ["uv", "export", "--no-hashes", "--python-preference", "only-system",
         "-o", "/content/requirements.lock.txt"],
        capture_output=True, text=True,
    )
    if r2.returncode == 0:
        subprocess.run(["pip", "install", "-q", "-r", "/content/requirements.lock.txt"], check=True)
        print("Окружение развёрнуто по uv.lock.")
    else:
        subprocess.run(
            ["pip", "install", "-q",
             "pandas>=2.2,<3.0", "numpy>=1.26", "scikit-learn>=1.5,<1.7",
             "pyyaml>=6.0", "joblib>=1.4", "dvc[s3]>=3.55,<4.0", "pandera>=0.20"],
            check=True,
        )
        print("Зависимости установлены через pip (uv export не удался).")
else:
    subprocess.run(
        ["pip", "install", "-q",
         "pandas>=2.2,<3.0", "numpy>=1.26", "scikit-learn>=1.5,<1.7",
         "pyyaml>=6.0", "joblib>=1.4", "dvc[s3]>=3.55,<4.0", "pandera>=0.20"],
        check=True,
    )
    print("Зависимости установлены через pip (uv.lock не создан).")

pyproject.toml записан.
uv lock: OK
Окружение развёрнуто по uv.lock.


## 6. pre-commit конфигурация

Аналогичным образом создает конфигурационный файл .pre-commit-config.yaml.Внутри него описываются внешние репозитории с проверками, версии этих репозиториев и конкретные hooks, то есть небольшие инструменты, которые выполняют определённые действия над файлами. Фиксация версий происходит через параметр rev для воспроизводимости.

2. Проверки Ruff. Первый подключённый репозиторий:https://github.com/astral-sh/ruff-pre-commit. В конфигурации включены два hook: ruff-check -- Проверяет на наличие ошибок и нарушения выбранных правил качества. Дополнительно передаётся --fix, позволяющий автоматически исправлять часть найденных проблем. ruff-format отвечает за единообразное форматирование кода. Приводит отступы, пробелы и расположение элементов к согласованному стилю.

3. Проверки из pre-commit-hooks. Второй подключённый репозиторий: https://github.com/pre-commit/pre-commit-hooks. Из него следующие проверки:
* check-added-large-files. -- Проверяет, не добавляются ли в Git слишком большие файлы. Задаем порог в 1024 КБ.
* check-yaml -- Проверяет синтаксическую корректность YAML-файлов (для конфигурации зависимостей и параметров эксперимента)
* detect-private-key -- Проверяет добавляемые файлы на наличие ключей.



In [ ]:
from pathlib import Path

Path(".pre-commit-config.yaml").write_text(
    "repos:\n"
    "  - repo: https://github.com/astral-sh/ruff-pre-commit\n"
    "    rev: v0.11.13\n"
    "    hooks:\n"
    "      - id: ruff-check\n"
    "        args: [--fix]\n"
    "      - id: ruff-format\n"
    "  - repo: https://github.com/pre-commit/pre-commit-hooks\n"
    "    rev: v5.0.0\n"
    "    hooks:\n"
    "      - id: check-added-large-files\n"
    "        args: ['--maxkb=1024']\n"
    "      - id: check-yaml\n"
    "      - id: detect-private-key\n",
    encoding="utf-8",
)
print(".pre-commit-config.yaml записан.")

.pre-commit-config.yaml записан.


## 7. params.yaml

На этом этапе создаётся файл `params.yaml`, в котором централизованно хранятся параметры подготовки данных, обучения модели и оценки её качества. Что позволяет менять настройки эксперимента без изменения логики Python-модулей. Кроме того, конфигурационный файл можно хранить в Git и отслеживать историю изменений вместе с кодом.

1. Создание файла. Для записи используется формат YAML — текстовый формат конфигурации, в котором данные представлены в виде пар «ключ — значение» и вложенных структур.
- prepare — настройки подготовки и разделения данных;
- train — параметры обучения модели;
- evaluate — параметры оценки качества.
2. В секции prepare находится параметр test_fraction: 0.2 для тестовой выборки. Оставшиеся примерно 80% используются для обучения.
3. В секции train задаются настройки, относящиеся к выбору и обучению модели.
Параметр model: hist_gb определяет, какой алгоритм будет использоваться в текущем эксперименте.
В конфигурации предусмотрены три варианта:
- linear_ridge — линейная регрессия с L2-регуляризацией;
- random_forest — ансамбль деревьев решений Random Forest;
- hist_gb — градиентный бустинг на основе гистограмм.

Также присутствует параметр target_transform: log1p, который описывает используемое преобразование целевой переменной shares. Преобразование `log1p(x)` вычисляет натуральный логарифм от 1 + x (для целевых переменных с сильно скошенным распределением и длинным правым хвостом). В данном проекте количество репостов у новостных статей может сильно различаться, логарифмическое преобразование помогает уменьшить влияние таких экстремально больших значений при обучении.

4. В подразделе linear_ridge находятся параметры линейной модели
- alpha: 10.0 — коэффициент регуляризации Ridge, управляет силой штрафа за большие значения коэффициентов модели. Увеличение alpha сильнее ограничивает коэффициенты и может уменьшить переобучение.
- select_k: 20 — количество признаков, которые будут оставлены при использовании отбора признаков. Если установить select_k: 0, отбор признаков отключается.
5. Настройки Random Forest, в котором описываются параметры ансамбля деревьев решений:
- n_estimators: 300 — количество деревьев в ансамбле;
- max_depth: 20 — максимальная глубина каждого дерева;
- min_samples_leaf: 3 — минимальное число объектов, которое должно находиться в листе дерева;
- random_state: 42 — фиксированное начальное состояние генератора случайных чисел;
- n_jobs: -1 — использование всех доступных вычислительных ядер для параллельного обучения.

6. В подразделе hist_gb находятся параметры градиентного бустинга:
- max_iter: 400 — максимальное количество итераций построения последовательности деревьев;
- max_depth: 6 — максимальная глубина отдельных деревьев;
- learning_rate: 0.05 — скорость обучения, определяющая вклад каждого следующего дерева;

Градиентный бустинг строит ансамбль последовательно: каждая следующая итерация пытается уменьшить ошибку уже построенной модели.
В текущей конфигурации именно `hist_gb` выбран в качестве активной модели.

7. Параметры оценки модели. evaluate: содержит параметр min_r2: 0.0, которое задает минимально допустимое значение коэффициента детерминации R^2. Коэффициент R^2 характеризует, насколько хорошо модель объясняет изменчивость целевой переменной относительно простого базового прогноза. Значение около 1 соответствует хорошему объяснению вариации целевой переменной. Значение 0 означает, что модель по этому показателю не превосходит базовый прогноз средним значением целевой переменной. Отрицательное значение означает, что модель работает хуже такого базового прогноза.

In [ ]:
from pathlib import Path

Path("params.yaml").write_text(
    "prepare:\n"
    "  test_fraction: 0.2\n"
    "\n"
    "train:\n"
    "  model: hist_gb # linear_ridge | random_forest | hist_gb\n"
    "  target_transform: log1p\n"
    "\n"
    "  linear_ridge:\n"
    "    alpha: 10.0\n"
    "    select_k: 20 # 0 — без отбора\n"
    "\n"
    "  random_forest:\n"
    "    n_estimators: 300\n"
    "    max_depth: 20\n"
    "    min_samples_leaf: 3\n"
    "    random_state: 42\n"
    "    n_jobs: -1\n"
    "\n"
    "  hist_gb:\n"
    "    max_iter: 400\n"
    "    max_depth: 6\n"
    "    learning_rate: 0.05\n"
    "    random_state: 42\n"
    "\n"
    "evaluate:\n"
    "  min_r2: 0.0\n",
    encoding="utf-8",
)
print("params.yaml записан.")

params.yaml записан.


## 8. src/schema.py (валидация через pandera)

src/schema.py описывает правила проверки исходного набора данных Online News Popularity. Для валидации используется библиотека Pandera. Проверка схемы нужна, чтобы обнаруживать некорректные данные до начала их обработки и обучения модели. Если в датасете отсутствуют обязательные столбцы, значения имеют неправильные типы или нарушают заданные ограничения, проблема будет выявлена на раннем этапе конвейера.

1. Создаёт схему RAW_SCHEMA. В ней определены два обязательных столбца:
   * timedelta — должен иметь числовой тип float, а его значения должны быть неотрицательными (Check.ge(0)).
   * shares — должен иметь тип int, а значения также должны быть неотрицательными.
   * strict=False означает, что в исходной таблице разрешены дополнительные столбцы, не перечисленные в схеме.
   * coerce=True разрешает Pandera приводить данные к указанным типам, если такое преобразование возможно.

2. Определяет функцию validate_raw(frame), которая принимает DataFrame и проверяет его с помощью RAW_SCHEMA.validate(frame, lazy=True). Параметр lazy=True позволяет собрать информацию о нескольких ошибках проверки, а не останавливаться после первой обнаруженной проблемы.

3. Дополнительно проверяет типы остальных столбцов. После проверки схемы функция проходит по всем столбцам таблицы. Столбец url исключён из этой проверки, поскольку он может содержать текстовые адреса страниц. Все остальные столбцы должны быть числовыми. Если обнаружены нечисловые столбцы, формируется список и вызывается ошибка с описанием проблемы.

5. Если проверки пройдены успешно, функция возвращает validated.



In [ ]:
from pathlib import Path

Path("src/schema.py").write_text(
    '"""Схема данных Online News Popularity (pandera >=0.20)."""\n'
    'from __future__ import annotations\n'
    '\n'
    'import pandas as pd\n'
    '# pandera >=0.20: старый корневой API удалён, используем pandera.pandas.\n'
    'import pandera.pandas as pa\n'
    'from pandera.pandas import Check, Column, DataFrameSchema\n'
    '\n'
    '\n'
    'RAW_SCHEMA = DataFrameSchema(\n'
    '    {\n'
    '        "timedelta": Column(float, Check.ge(0)),\n'
    '        "shares": Column(int, Check.ge(0)),\n'
    '    },\n'
    '    strict=False,\n'
    '    coerce=True,\n'
    ')\n'
    '\n'
    '\n'
    'def validate_raw(frame: pd.DataFrame) -> pd.DataFrame:\n'
    '    """Проверяет обязательные колонки и числовой тип остальных."""\n'
    '    validated = RAW_SCHEMA.validate(frame, lazy=True)\n'
    '    non_numeric = [\n'
    '        c for c in validated.columns\n'
    '        if c not in {"url"} and not pd.api.types.is_numeric_dtype(validated[c])\n'
    '    ]\n'
    '    if non_numeric:\n'
    '        raise ValueError(f"Нечисловые колонки: {non_numeric}")\n'
    '    return validated\n',
    encoding="utf-8",
)
print("src/schema.py записан.")

src/schema.py записан.


## 9. src/prepare.py, который отвечает за подготовку исходного датасета перед обучением модели.

1. Определяет функцию split_by_time(data, test_fraction). Эта функция разделяет данные на обучающую и тестовую выборки с учётом столбца timedelta. timedelta показывает количество часов от первой статьи, причём большее значение соответствует более старой статье. Сначала функция получает отсортированный список уникальных значений timedelta, затем рассчитывает размер тестовой выборки. Далее данные разделяются:
   - test — статьи с timedelta <= cutoff, то есть более новые;
   - train — статьи с timedelta > cutoff, то есть более старые.
2. Определяет основную функцию prepare_data(). Сначала функция открывает YAML-файл и получает параметр prepare.test_fraction. Значение преобразуется в float.
3. Загружает и проверяет исходный датасет, через pd.read_csv(input_path).   Команда raw.columns.str.strip() удаляет лишние пробелы в начале и конце названий столбцов. Это помогает избежать ошибок при обращении к колонкам по имени. После этого вызывается validate_raw(raw), которая проверяет данные по определенным выше правилам.
4. Функция split_by_time() возвращает train и test. Затем проводятся еще дополнительные проверки: обе выборки должны быть непустыми и минимальное значение timedelta в обучающей выборке должно быть больше максимального значения timedelta в тестовой.
5. Сохраняет подготовленные данные. Если каталог data/processed отсутствует, он создаётся с помощью mkdir(parents=True, exist_ok=True). Затем сохраняются два файла: data/processed/train.csv и data/processed/test.csv. В конце выводится количество обработанных строк, размеры выборок и значение временной границы.



In [ ]:
from pathlib import Path

Path("src/prepare.py").write_text(
    '"""Подготовка данных: проверка схемы и разбиение по времени.\n'
    '\n'
    'timedelta — часы от первой статьи; большее значение == более старая статья.\n'
    'Тест — самые новые статьи (меньший timedelta).\n'
    '"""\n'
    'from __future__ import annotations\n'
    '\n'
    'from pathlib import Path\n'
    '\n'
    'import pandas as pd\n'
    'import yaml\n'
    '\n'
    'from src.schema import validate_raw\n'
    '\n'
    '\n'
    'def split_by_time(\n'
    '    data: pd.DataFrame, test_fraction: float\n'
    ') -> tuple[pd.DataFrame, pd.DataFrame]:\n'
    '    """Делит по уникальным timedelta, чтобы точка не попала в обе выборки."""\n'
    '    times = sorted(data["timedelta"].unique())\n'
    '    target_test = int(len(data) * test_fraction)\n'
    '    cutoff = min(\n'
    '        times,\n'
    '        key=lambda v: abs(int((data["timedelta"] <= v).sum()) - target_test),\n'
    '    )\n'
    '    test = data[data["timedelta"] <= cutoff].copy()\n'
    '    train = data[data["timedelta"] > cutoff].copy()\n'
    '    return train, test\n'
    '\n'
    '\n'
    'def prepare_data(\n'
    '    input_path: str = "data/external/OnlineNewsPopularity.csv",\n'
    '    output_dir: str = "data/processed",\n'
    '    params_path: str = "params.yaml",\n'
    ') -> None:\n'
    '    with open(params_path, encoding="utf-8") as fh:\n'
    '        params = yaml.safe_load(fh)["prepare"]\n'
    '    test_fraction = float(params["test_fraction"])\n'
    '    if not 0 < test_fraction < 1:\n'
    '        raise ValueError("test_fraction должен быть между 0 и 1.")\n'
    '\n'
    '    raw = pd.read_csv(input_path)\n'
    '    raw.columns = raw.columns.str.strip()\n'
    '    data = validate_raw(raw)\n'
    '\n'
    '    train, test = split_by_time(data, test_fraction)\n'
    '\n'
    '    assert not train.empty and not test.empty\n'
    '    assert train["timedelta"].min() > test["timedelta"].max()\n'
    '\n'
    '    out = Path(output_dir)\n'
    '    out.mkdir(parents=True, exist_ok=True)\n'
    '    train.to_csv(out / "train.csv", index=False)\n'
    '    test.to_csv(out / "test.csv", index=False)\n'
    '\n'
    '    cutoff_value = float(train["timedelta"].min())\n'
    '    print(f"Схема пройдена: {len(data)} строк")\n'
    '    print(\n'
    '        f"Train: {len(train)} | Test: {len(test)} "\n'
    '        f"| cutoff timedelta = {cutoff_value}"\n'
    '    )\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    prepare_data()\n',
    encoding="utf-8",
)
print("src/prepare.py записан (совместим с ruff).")

src/prepare.py записан (совместим с ruff).


## 10. src/featurize.py

Модуль src/featurize.py предназначен для формирования признаков и целевой переменной, которые будут использоваться при обучении модели. На предыдущем этапе исходный датасет был разделён на train и test. Теперь необходимо отделить входные признаки от целевой переменной shares, проверить корректность полученных данных и сохранить их в отдельные CSV-файлы

1. Для этого задаются две константы:
   - TARGET = "shares" — целевая переменная, которую должна предсказывать модель;
   - DROP = {url} — набор столбцов, которые необходимо исключить из признаков.

2. Определяем функцию featurize(), которая работает с каталогами processed (подготовленные обучающие и тестовые данные) и data/features для сохранения сформированных признаков.
3. Загружает обучающую и тестовую выборки.
4. Проверяет наличие целевой переменной. Перед разделением данных убеждаемся, что столбец shares присутствует и в обучающей, и в тестовой выборке.
5. Формирует список признаков. Список feature_cols строится по столбцам обучающей выборки. Из него исключаются:
   - столбцы, входящие в DROP;
   - целевая переменная TARGET.
   Аналогичным образом формируется test_cols для тестовой выборки.
   Затем сравниваются созданные списки. Если они не совпадают, выполнение прекращается с ошибкой, поскольку модель должна получать одинаковый набор признаков при обучении и тестировании.
6. Создаются четыре объекта:
   - x_train — признаки обучающей выборки;
   - x_test — признаки тестовой выборки;
   - y_train — значения shares для обучения;
   - y_test — значения shares для тестирования.
7. Проверяет корректность признаков, поскольку все признаки должны иметь числовой тип, и не должно быть пропущенных значений.
8. В каталог data/features записываются четыре файла:
   - X_train.csv — обучающие признаки;
   - X_test.csv — тестовые признаки;
   - y_train.csv — целевая переменная для обучения;
   - y_test.csv — целевая переменная для тестирования.


In [ ]:
from pathlib import Path

Path("src/featurize.py").write_text(
    '"""Признаки: убираем url и неинформативные колонки, оставляем числовые."""\n'
    'from __future__ import annotations\n'
    '\n'
    'from pathlib import Path\n'
    '\n'
    'import pandas as pd\n'
    '\n'
    'TARGET = "shares"\n'
    'DROP = {"url"}\n'
    '\n'
    '\n'
    'def featurize(\n'
    '    processed_dir: str = "data/processed",\n'
    '    features_dir: str = "data/features",\n'
    ') -> None:\n'
    '    proc = Path(processed_dir)\n'
    '    feat = Path(features_dir)\n'
    '    feat.mkdir(parents=True, exist_ok=True)\n'
    '\n'
    '    train = pd.read_csv(proc / "train.csv")\n'
    '    test = pd.read_csv(proc / "test.csv")\n'
    '\n'
    '    if TARGET not in train.columns or TARGET not in test.columns:\n'
    '        raise ValueError("Целевая колонка shares отсутствует.")\n'
    '\n'
    '    feature_cols = [c for c in train.columns if c not in DROP and c != TARGET]\n'
    '    test_cols = [c for c in test.columns if c not in DROP and c != TARGET]\n'
    '    if feature_cols != test_cols:\n'
    '        raise ValueError("Наборы признаков train и test не совпадают.")\n'
    '\n'
    '    x_train, x_test = train[feature_cols], test[feature_cols]\n'
    '    y_train, y_test = train[TARGET], test[TARGET]\n'
    '\n'
    '    for name, frame in (("X_train", x_train), ("X_test", x_test)):\n'
    '        non_num = [\n'
    '            c for c in frame.columns\n'
    '            if not pd.api.types.is_numeric_dtype(frame[c])\n'
    '        ]\n'
    '        if non_num:\n'
    '            raise ValueError(f"{name}: нечисловые признаки {non_num}")\n'
    '        if frame.isna().any().any():\n'
    '            raise ValueError(f"{name}: пропуски недопустимы.")\n'
    '\n'
    '    x_train.to_csv(feat / "X_train.csv", index=False)\n'
    '    x_test.to_csv(feat / "X_test.csv", index=False)\n'
    '    y_train.to_frame().to_csv(feat / "y_train.csv", index=False)\n'
    '    y_test.to_frame().to_csv(feat / "y_test.csv", index=False)\n'
    '\n'
    '    print(f"Признаков: {len(feature_cols)}")\n'
    '    print(f"X_train {x_train.shape} | X_test {x_test.shape}")\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    featurize()\n',
    encoding="utf-8",
)
print("src/featurize.py записан (совместим с ruff).")

src/featurize.py записан (совместим с ruff).


## 11. src/train.py

Создаем модуль, который отвечает за обучение модели машинного обучения для прогнозирования популярности новостных статей. В отличие от предыдущих модулей, которые подготавливали данные, здесь реализуется непосредственно процесс обучения. Выбор алгоритма и его гиперпараметры определяются через файл params.yaml.

1. Сначала определяем функцию make_model(train_params). Эта функция отвечает за создание модели в соответствии с train.model из файла params.yaml. Сначала считывается название алгоритма name = train_params["model"], затем в зависимости от значения name создаётся соответствующий объект модели.
2. Реализует вариант linear_ridge. Формируется последовательность шагов:
   - StandardScaler() — стандартизация признаков;
   - SelectKBest(f_regression, k=k) — необязательный отбор наиболее информативных признаков;
   - Ridge(alpha=...) — сама модель Ridge-регрессии (метод построения линейной регрессионной модели с использованием L2-регуляризации.)
3. Реализует вариант random_forest. Если в конфигурации выбрана модель random_forest, создаётся RandomForestRegressor. Используемые параметры:
   - n_estimators — количество деревьев;
   - max_depth — максимальная глубина деревьев;
   - min_samples_leaf — минимальное количество объектов в листе;
   - random_state — фиксированное значение для воспроизводимости;
   - n_jobs — количество используемых потоков выполнения. Значение -1 означает использование всех доступных процессоров.
4. Реализует вариант hist_gb. Если выбрана модель hist_gb, создаётся HistGradientBoostingRegressor. Его параметры:
   - max_iter — количество итераций бустинга;
   - max_depth — максимальная глубина деревьев;
   - learning_rate — скорость обучения;
   - random_state — параметр воспроизводимости.
5. Обрабатывает неизвестные названия моделей. Если значение train.model не соответствует ни одному из трёх поддерживаемых вариантов, вызывается ValueError.
6. Определяет основную функцию train(). Функция принимает три параметра:
   - features_dir — каталог с подготовленными признаками;
   - model_path — путь для сохранения модели;
   - params_path — путь к конфигурационному файлу.
По умолчанию используются data/features, models/model.joblib и params.yaml.
7. Загружает параметры обучения. Из YAML-файла считывается раздел train.   Именно он определяет используемый алгоритм и его гиперпараметры.
8. Загружает обучающие данные. Считываются:
   - X_train.csv — матрица признаков;
   - y_train.csv — целевая переменная.
Метод .squeeze("columns") преобразует таблицу с одним столбцом в одномерную структуру, подходящую для передачи в scikit-learn.
9. Выполняет проверки перед обучением. Перед вызовом fit() проверяется:
    - совпадает ли количество строк в X_train и y_train;
    - отсутствуют ли пропущенные значения;
    - нет ли отрицательных значений в shares.
10. Создаёт и обучает модель. Сначала вызывается make_model(params), после чего модель обучается: model.fit(X_train, np.log1p(y_train)). Здесь используется преобразование log1p, то есть логарифм от 1 + shares. Такое преобразование применяется для уменьшения влияния очень больших значений целевой переменной и стабилизации обучения при наличии длинного правого хвоста распределения популярности.
11. Сохраняет обученную модель. Путь сохранения задаётся параметром model_path. Если родительский каталог отсутствует, он создаётся через mkdir(parents=True, exist_ok=True). Затем модель сохраняется с помощью: joblib.dump(model, out). По умолчанию результат записывается в models/model.joblib.




In [ ]:
from pathlib import Path

Path("src/train.py").write_text(
    '"""Обучение модели. Модель выбирается в params.yaml → train.model."""\n'
    'from __future__ import annotations\n'
    '\n'
    'from pathlib import Path\n'
    '\n'
    'import joblib\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import yaml\n'
    'from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor\n'
    'from sklearn.feature_selection import SelectKBest, f_regression\n'
    'from sklearn.linear_model import Ridge\n'
    'from sklearn.pipeline import make_pipeline\n'
    'from sklearn.preprocessing import StandardScaler\n'
    '\n'
    '\n'
    'def make_model(train_params: dict):\n'
    '    name = train_params["model"]\n'
    '    if name == "linear_ridge":\n'
    '        cfg = train_params.get("linear_ridge", {})\n'
    '        steps = [StandardScaler()]\n'
    '        k = int(cfg.get("select_k", 0))\n'
    '        if k > 0:\n'
    '            steps.append(SelectKBest(f_regression, k=k))\n'
    '        steps.append(Ridge(alpha=float(cfg.get("alpha", 10.0))))\n'
    '        return make_pipeline(*steps)\n'
    '    if name == "random_forest":\n'
    '        cfg = train_params.get("random_forest", {})\n'
    '        return RandomForestRegressor(\n'
    '            n_estimators=int(cfg.get("n_estimators", 300)),\n'
    '            max_depth=int(cfg.get("max_depth", 20)),\n'
    '            min_samples_leaf=int(cfg.get("min_samples_leaf", 3)),\n'
    '            random_state=int(cfg.get("random_state", 42)),\n'
    '            n_jobs=int(cfg.get("n_jobs", -1)),\n'
    '        )\n'
    '    if name == "hist_gb":\n'
    '        cfg = train_params.get("hist_gb", {})\n'
    '        return HistGradientBoostingRegressor(\n'
    '            max_iter=int(cfg.get("max_iter", 400)),\n'
    '            max_depth=int(cfg.get("max_depth", 6)),\n'
    '            learning_rate=float(cfg.get("learning_rate", 0.05)),\n'
    '            random_state=int(cfg.get("random_state", 42)),\n'
    '        )\n'
    '    raise ValueError(f"Неизвестная модель: {name!r}")\n'
    '\n'
    '\n'
    'def train(\n'
    '    features_dir: str = "data/features",\n'
    '    model_path: str = "models/model.joblib",\n'
    '    params_path: str = "params.yaml",\n'
    ') -> None:\n'
    '    with open(params_path, encoding="utf-8") as fh:\n'
    '        params = yaml.safe_load(fh)["train"]\n'
    '\n'
    '    feat = Path(features_dir)\n'
    '    X_train = pd.read_csv(feat / "X_train.csv")\n'
    '    y_train = pd.read_csv(feat / "y_train.csv").squeeze("columns")\n'
    '\n'
    '    if len(X_train) != len(y_train):\n'
    '        raise ValueError("Размеры X_train и y_train не совпадают.")\n'
    '    if X_train.isna().any().any() or y_train.isna().any():\n'
    '        raise ValueError("Пропуски в данных недопустимы.")\n'
    '    if (y_train < 0).any():\n'
    '        raise ValueError("shares не может быть отрицательным.")\n'
    '\n'
    '    model = make_model(params)\n'
    '    # log1p стабилизирует тяжёлый хвост shares\n'
    '    model.fit(X_train, np.log1p(y_train))\n'
    '\n'
    '    out = Path(model_path)\n'
    '    out.parent.mkdir(parents=True, exist_ok=True)\n'
    '    joblib.dump(model, out)\n'
    '\n'
    '    print(f"Модель: {params[\"model\"]}")\n'
    '    print(f"Обучено на {len(X_train)} статьях, {X_train.shape[1]} признаках.")\n'
    '    print(f"Сохранено: {out}")\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    train()\n',
    encoding="utf-8",
)
print("src/train.py записан.")

src/train.py записан.


## 12. src/evaluate.py

Создаёт модуль src/evaluate.py, который оценивает обученную модель на тестовой выборке и сравнивает её с простой базовой моделью.

1. Подключает joblib, numpy, pandas, yaml, json и метрики из sklearn.metrics.
2. Функция _metrics(y_true, y_pred) рассчитывает три показателя:
   - MAE — среднюю абсолютную ошибку;
   - RMSE — корень из средней квадратичной ошибки;
   - R² — коэффициент детерминации.
3. Функция evaluate() загружает параметры из params.yaml, включая минимально допустимый evaluate.min_r2.
4. Загружает обучающие и тестовые признаки и целевые значения, проверяет совпадение колонок.
5. Загружает модель из models/model.joblib и получает прогнозы. Поскольку обучение выполнялось на log1p(shares), результат возвращается в исходный масштаб через expm1. np.maximum(0, ...) не позволяет получить отрицательное число репостов.
6. Рассчитывает baseline: константный прогноз, полученный из среднего значения log1p(shares) на обучающей выборке.
7. Сохраняет метрики модели и baseline в reports/metrics.json. JSON имеет плоскую структуру с числовыми значениями — такой формат удобен для DVC.
8. Если model_r2 оказывается ниже min_r2, функция вызывает ValueError.


In [ ]:
from pathlib import Path

Path("src/evaluate.py").write_text(
    '"""Оценка модели: метрики, baseline и проверка порога."""\n'
    'from __future__ import annotations\n'
    '\n'
    'import json\n'
    'from pathlib import Path\n'
    '\n'
    'import joblib\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import yaml\n'
    'from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n'
    '\n'
    '\n'
    'def _metrics(y_true, y_pred) -> dict:\n'
    '    return {\n'
    '        "mae":  float(mean_absolute_error(y_true, y_pred)),\n'
    '        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),\n'
    '        "r2":   float(r2_score(y_true, y_pred)),\n'
    '    }\n'
    '\n'
    '\n'
    'def evaluate(\n'
    '    features_dir: str = "data/features",\n'
    '    model_path: str   = "models/model.joblib",\n'
    '    report_path: str  = "reports/metrics.json",\n'
    '    params_path: str  = "params.yaml",\n'
    ') -> None:\n'
    '    with open(params_path, encoding="utf-8") as fh:\n'
    '        params = yaml.safe_load(fh)\n'
    '    min_r2 = float(params.get("evaluate", {}).get("min_r2", 0.0))\n'
    '\n'
    '    feat = Path(features_dir)\n'
    '    X_train = pd.read_csv(feat / "X_train.csv")\n'
    '    y_train = pd.read_csv(feat / "y_train.csv").squeeze("columns")\n'
    '    X_test  = pd.read_csv(feat / "X_test.csv")\n'
    '    y_test  = pd.read_csv(feat / "y_test.csv").squeeze("columns")\n'
    '\n'
    '    if list(X_train.columns) != list(X_test.columns):\n'
    '        raise ValueError("Признаки train и test не совпадают.")\n'
    '\n'
    '    model = joblib.load(model_path)\n'
    '    pred = np.maximum(0, np.expm1(model.predict(X_test)))\n'
    '\n'
    '    # Baseline в том же пространстве, что и предсказание модели:\n'
    '    # среднее log1p(shares) на train, возвращённое через expm1.\n'
    '    baseline_value = float(np.expm1(np.log1p(y_train).mean()))\n'
    '    baseline = np.full(len(y_test), baseline_value)\n'
    '\n'
    '    m = _metrics(y_test, pred)\n'
    '    b = _metrics(y_test, baseline)\n'
    '\n'
    '    # ПЛОСКИЙ JSON: DVC читает только плоские ключи и числа.\n'
    '    results = {\n'
    '        "test_size":            int(len(y_test)),\n'
    '        "n_features":           int(X_test.shape[1]),\n'
    '        "baseline_mean_shares": baseline_value,\n'
    '        "model_mae":   m["mae"],\n'
    '        "model_rmse":  m["rmse"],\n'
    '        "model_r2":    m["r2"],\n'
    '        "baseline_mae":  b["mae"],\n'
    '        "baseline_rmse": b["rmse"],\n'
    '        "baseline_r2":   b["r2"],\n'
    '    }\n'
    '\n'
    '    report = Path(report_path)\n'
    '    report.parent.mkdir(parents=True, exist_ok=True)\n'
    '    report.write_text(\n'
    '        json.dumps(results, indent=2, ensure_ascii=False) + "\\n",\n'
    '        encoding="utf-8",\n'
    '    )\n'
    '\n'
    '    print("Модель:")\n'
    '    print(f"  MAE: {m[\'mae\']:.4f}")\n'
    '    print(f"  RMSE: {m[\'rmse\']:.4f}")\n'
    '    print(f"  R2: {m[\'r2\']:.4f}")\n'
    '    print("Baseline (среднее log1p(shares) на train):")\n'
    '    print(f"  MAE: {b[\'mae\']:.4f}")\n'
    '    print(f"  RMSE: {b[\'rmse\']:.4f}")\n'
    '    print(f"  R2: {b[\'r2\']:.4f}")\n'
    '    print(f"Отчёт: {report}")\n'
    '\n'
    '    if m["r2"] < min_r2:\n'
    '        raise ValueError(f"R2 = {m[\'r2\']:.4f} ниже порога {min_r2}")\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    evaluate()\n',
    encoding="utf-8",
)
print("src/evaluate.py записан (плоский JSON для DVC).")

src/evaluate.py записан (плоский JSON для DVC).


## 13. tests/test_smoke.py

Создаёт файл tests/test_smoke.py с простыми smoke-тестами. Они проверяют, что основные модули проекта существуют, а файл с метриками имеет ожидаемую структуру.

1. test_src_modules_exist() перебирает модули prepare, featurize, train, evaluate и schema. Для каждого проверяет наличие соответствующего файла в каталоге src.

2. test_metrics_file_has_keys() проверяет наличие reports/metrics.json. Если файла пока нет, тест завершается без ошибки — это предусмотрено для запуска до первого dvc repro. Если файл существует, он читается как JSON. Затем проверяется наличие шести ключей: model_mae, model_rmse, model_r2,baseline_mae, baseline_rmse и baseline_r2.



In [ ]:
from pathlib import Path

Path("tests/test_smoke.py").write_text(
    '"""Smoke-тесты: модули на месте, метрики содержат ключи."""\n'
    'import json\n'
    'from pathlib import Path\n'
    '\n'
    '\n'
    'def test_src_modules_exist():\n'
    '    for mod in ("prepare", "featurize", "train", "evaluate", "schema"):\n'
    '        assert Path(f"src/{mod}.py").exists()\n'
    '\n'
    '\n'
    'def test_metrics_file_has_keys():\n'
    '    p = Path("reports/metrics.json")\n'
    '    if not p.exists():\n'
    '        return  # до первого dvc repro\n'
    '    m = json.loads(p.read_text(encoding="utf-8"))\n'
    '    expected = {\n'
    '        "model_mae", "model_rmse", "model_r2",\n'
    '        "baseline_mae", "baseline_rmse", "baseline_r2",\n'
    '    }\n'
    '    assert expected <= set(m)\n',
    encoding="utf-8",
)
print("tests/test_smoke.py записан.")

tests/test_smoke.py записан.


## 14. Скачивание датасета

Загружает исходный датасет Online News Popularity с сайта UCI, распаковывает архив и проверяет базовые характеристики CSV-файла.

1. Импортирует Path, urlretrieve, ZipFile и pandas.
2. Определяет путь к архиву и итоговому файлу data/external/OnlineNewsPopularity.csv.
3. Перед загрузкой удаляет старые архив и CSV, если они существуют. Благодаря этому скачивание начинается заново, а не использует потенциально устаревшую копию.
4. Загружает архив по адресу UCI с помощью urlretrieve().
5. Открывает ZIP-архив и ищет внутри файл OnlineNewsPopularity.csv.
6. Извлекает CSV в data/external.
7. Загружает таблицу через pandas, очищает названия колонок от лишних пробелов и выводит количество строк, минимальное и максимальное значения timedelta, а также количество отрицательных значений этого признака.
8. Проверяет два условия через assert: в датасете должно быть ровно 39 644 строки, а отрицательных значений timedelta быть не должно.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import pandas as pd

archive = Path("/content/online-news-popularity.zip")
csv_out = Path("data/external/OnlineNewsPopularity.csv")
url = "https://archive.ics.uci.edu/static/public/332/online+news+popularity.zip"

# Всегда с нуля
if archive.exists():
    archive.unlink()
if csv_out.exists():
    csv_out.unlink()

print("Скачиваем архив (~20 МБ)...")
urlretrieve(url, archive)
print("Распаковываем...")
with ZipFile(archive) as z:
    members = [n for n in z.namelist() if n.endswith("OnlineNewsPopularity.csv")]
    if not members:
        raise FileNotFoundError("CSV не найден в архиве.")
    with z.open(members[0]) as src, csv_out.open("wb") as dst:
        dst.write(src.read())

df = pd.read_csv(csv_out)
df.columns = df.columns.str.strip()
print(f"строк: {len(df)}")
print(f"min timedelta: {df['timedelta'].min()}")
print(f"max timedelta: {df['timedelta'].max()}")
print(f"отрицательных timedelta: {(df['timedelta'] < 0).sum()}")

assert len(df) == 39644, f"Ожидалось 39644 строки, получено {len(df)}"
assert (df["timedelta"] < 0).sum() == 0, "Есть отрицательные timedelta"
print("CSV чистый. Размер:", f"{csv_out.stat().st_size / 1024**2:.1f} МБ")

Скачиваем архив (~20 МБ)...
Распаковываем...
строк: 39644
min timedelta: 8.0
max timedelta: 731.0
отрицательных timedelta: 0
CSV чистый. Размер: 23.2 МБ


## 15. Инициализация DVC

Инициализирует DVC в репозитории и настраивает удалённое хранилище данных на базе DagsHub S3.

1. Импортирует subprocess и Path.
2. Определяет функцию run(cmd, check=True), которая запускает системную команду, перехватывает её вывод и при ошибке вызывает RuntimeError.
3. Проверяет наличие каталога .dvc. Если его нет, выполняет dvc init. Если DVC уже инициализирован, повторная инициализация пропускается.
4. Формирует адрес удалённого хранилища endpoint на основе DAGSHUB_USER и REPO_NAME.
5. Удаляет старый remote с именем origin, если он был настроен.
6. Создаёт новый remote origin с адресом s3://dvc и указывает endpoint DagsHub.
7. Настраивает access_key_id и secret_access_key через dvc remote modify --local. Для обоих значений используется DAGSHUB_TOKEN.

DVC позволяет хранить большие файлы данных и ML-артефакты отдельно от Git, но при этом отслеживать их версии. DagsHub выступает удаленным хранилищем, а локальная настройка токена позволяет не помещать секретные данные доступа в обычную конфигурацию репозитория.

In [ ]:
import subprocess
from pathlib import Path

def run(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(r.stderr)
    return r.stdout.strip()

if not Path(".dvc").exists():
    run(["dvc", "init"])
    print("DVC инициализирован.")
else:
    print("DVC уже инициализирован — пропускаем.")

endpoint = f"https://dagshub.com/{DAGSHUB_USER}/{REPO_NAME}.s3"

subprocess.run(["dvc", "remote", "remove", "origin"], capture_output=True)
run(["dvc", "remote", "add", "-d", "origin", "s3://dvc"])
run(["dvc", "remote", "modify", "origin", "endpointurl", endpoint])
run(["dvc", "remote", "modify", "--local", "origin", "access_key_id", DAGSHUB_TOKEN])
run(["dvc", "remote", "modify", "--local", "origin", "secret_access_key", DAGSHUB_TOKEN])
print(f"DVC remote: {endpoint}")


DVC инициализирован.
DVC remote: https://dagshub.com/IsamuYosida/lr1newspopularity.s3


## 16. .gitignore + данные под DVC

В этой секции выполняются две связанные задачи: настройка исключений Git и передача исходного датасета под управление DVC.

Код читает текущий .gitignore и проверяет, присутствуют ли необходимые правила. Если каких-то строк нет, добавляет их. Из Git исключаются:
- исходные и промежуточные CSV-данные;
- обработанные признаки;
- обученные модели;
- отчёты.
При этом правила !/data/external/*.dvc и !/data/raw/*.dvc разрешают Git отслеживать DVC-метафайлы, несмотря на исключение самих данных.

Вторая кодовая ячейка — DVC add и push
1. Выполняет dvc add -f data/external/OnlineNewsPopularity.csv. Команда добавляет датасет под управление DVC и создаёт метафайл .dvc, содержащий информацию о версии файла.
2. Выполняет dvc push, чтобы отправить данные в настроенное удалённое хранилище.
3. Проверяет код завершения каждой команды и вызывает ошибку, если операция завершилась неудачно.


In [ ]:
from pathlib import Path

gitignore = Path(".gitignore")
content = gitignore.read_text(encoding="utf-8") if gitignore.exists() else ""

additions = []
for line in [
    "/data/external/*",
    "!/data/external/*.dvc",
    "/data/raw/*",
    "!/data/raw/*.dvc",
    "/data/processed/",
    "/data/features/",
    "/models/",
    "/reports/",
]:
    if line not in content:
        additions.append(line)

if additions:
    content = content.rstrip("\n") + "\n" + "\n".join(additions) + "\n"
    gitignore.write_text(content, encoding="utf-8")
    print(f"Добавлено {len(additions)} строк в .gitignore")
else:
    print(".gitignore уже содержит все нужные строки")

print(gitignore.read_text(encoding="utf-8"))


.gitignore уже содержит все нужные строки

__pycache__/
*.pyc
.ruff_cache/
.pytest_cache/
.ipynb_checkpoints/
/data/external/*
!/data/external/*.dvc
/data/raw/*
!/data/raw/*.dvc
/data/processed/
/data/features/
/models/
/reports/



In [ ]:
import subprocess

r = subprocess.run(["dvc", "add", "-f", "data/external/OnlineNewsPopularity.csv"],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError(r.stderr)

r = subprocess.run(["dvc", "push"], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError(r.stderr)
print("Чистые данные v1 отправлены в DagsHub.")


To track the changes with git, run:

	git add data/external/OnlineNewsPopularity.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

Everything is up to date.

Чистые данные v1 отправлены в DagsHub.


## 17. dvc.yaml

Создаёт файл dvc.yaml, в котором описывается последовательность этапов обработки данных, обучения и оценки модели.

В файле определены четыре стадии:
1. prepare — запускает python -m src.prepare. Зависит от исходного CSV, src/prepare.py, src/schema.py и параметров prepare. Создаёт train.csv и test.csv.
2. featurize — запускает python -m src.featurize. Использует подготовленные выборки и создаёт четыре файла с X и y для train и test.
3. train — запускает python -m src.train. Использует обучающие признаки и параметры train, создаёт models/model.joblib.
4. evaluate — запускает python -m src.evaluate. Использует модель, обучающие и тестовые данные, а также параметры evaluate. Результатом становится reports/metrics.json.

Для каждой стадии явно указаны команды, зависимости (deps), параметры (params), выходные файлы (outs) или метрики (metrics).

dvc.yaml описывает конвейер как граф зависимостей. DVC может определить, какие этапы нужно пересчитать после изменения входных данных, кода или параметров.

In [ ]:
from pathlib import Path

Path("dvc.yaml").write_text(
    "stages:\n"
    "  prepare:\n"
    "    cmd: python -m src.prepare\n"
    "    deps:\n"
    "      - src/prepare.py\n"
    "      - src/schema.py\n"
    "      - data/external/OnlineNewsPopularity.csv\n"
    "    params:\n"
    "      - prepare\n"
    "    outs:\n"
    "      - data/processed/train.csv\n"
    "      - data/processed/test.csv\n"
    "\n"
    "  featurize:\n"
    "    cmd: python -m src.featurize\n"
    "    deps:\n"
    "      - src/featurize.py\n"
    "      - data/processed/train.csv\n"
    "      - data/processed/test.csv\n"
    "    outs:\n"
    "      - data/features/X_train.csv\n"
    "      - data/features/X_test.csv\n"
    "      - data/features/y_train.csv\n"
    "      - data/features/y_test.csv\n"
    "\n"
    "  train:\n"
    "    cmd: python -m src.train\n"
    "    deps:\n"
    "      - src/train.py\n"
    "      - data/features/X_train.csv\n"
    "      - data/features/y_train.csv\n"
    "    params:\n"
    "      - train\n"
    "    outs:\n"
    "      - models/model.joblib\n"
    "\n"
    "  evaluate:\n"
    "    cmd: python -m src.evaluate\n"
    "    deps:\n"
    "      - src/evaluate.py\n"
    "      - models/model.joblib\n"
    "      - data/features/X_train.csv\n"
    "      - data/features/y_train.csv\n"
    "      - data/features/X_test.csv\n"
    "      - data/features/y_test.csv\n"
    "    params:\n"
    "      - evaluate\n"
    "    metrics:\n"
    "      - reports/metrics.json:\n"
    "          cache: false\n",
    encoding="utf-8",
)
print("dvc.yaml записан.")

dvc.yaml записан.


## 18. Первый прогон пайплайна

Запускает полный конвейер DVC с принудительным пересчётом стадий.

1. Использует subprocess.run() для выполнения команды dvc repro -f.
2. Параметр -f заставляет DVC выполнить стадии заново, даже если он считает их актуальными.
3. Вывод команды сохраняется в r.stdout и печатается на экран.
4. Если команда завершилась с ошибкой, дополнительно выводится stderr (ошибка с отладочными данными).

Во время выполнения последовательно запускаются стадии prepare, featurize, train и evaluate. В выводе можно увидеть размеры выборок, количество признаков, выбранную модель и рассчитанные метрики.

Первый запуск проверяет, что все компоненты конвейера связаны правильно и могут работать последовательно. DVC также создаёт или обновляет dvc.lock, фиксируя информацию о зависимостях и результатах выполнения стадий.

In [ ]:
import subprocess

r = subprocess.run(["dvc", "repro", "-f"], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise RuntimeError("dvc repro завершился с ошибкой.")

Running stage 'prepare':
> python -m src.prepare
Схема пройдена: 39644 строк
Train: 31693 | Test: 7951 | cutoff timedelta = 133.0
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

Running stage 'featurize':
> python -m src.featurize
Признаков: 59
X_train (31693, 59) | X_test (7951, 59)
Updating lock file 'dvc.lock'

Running stage 'train':
> python -m src.train
Модель: hist_gb
Обучено на 31693 статьях, 59 признаках.
Сохранено: models/model.joblib
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python -m src.evaluate
Модель:
  MAE: 2087.5341
  RMSE: 8128.6478
  R2: 0.0012
Baseline (среднее log1p(shares) на train):
  MAE: 2216.3654
  RMSE: 8224.0042
  R2: -0.0224
Отчёт: reports/metrics.json
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.



## 19. Второй прогон + dag

Проверяет, что DVC умеет распознавать неизменившиеся стадии и не выполнять лишнюю работу. Дополнительно строит граф зависимостей пайплайна.

1. Выполняет dvc repro
2. Печатает вывод команды. Если входные данные, код и параметры не изменились, DVC пропускает соответствующие стадии. В выводе это отображается сообщением "didn't change, skipping".
3. Затем запускает dvc dag.
4. Печатает построенный DVC граф, показывающий связи между prepare, featurize, train и evaluate.

Повторный запуск демонстрирует механизм инкрементального выполнения: пересчитываются только те этапы, для которых изменились зависимости. Это экономит время и вычислительные ресурсы. Команда dvc dag помогает визуально проверить структуру и порядок стадий конвейера.

In [ ]:
import subprocess

r = subprocess.run(["dvc", "repro"], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)

print("\nГраф зависимостей")
r2 = subprocess.run(["dvc", "dag"], capture_output=True, text=True)
print(r2.stdout)


Stage 'prepare' didn't change, skipping
Stage 'featurize' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.


Граф зависимостей
         +---------+      
         | prepare |      
         +---------+      
              *           
              *           
              *           
        +-----------+     
        | featurize |     
        +-----------+     
         **        **     
       **            *    
      *               **  
+-------+               * 
| train |             **  
+-------+            *    
         **        **     
           **    **       
             *  *         
        +----------+      
        | evaluate |      
        +----------+      



## 20. Метрики

Читает и выводит содержимое отчёта reports/metrics.json, сформированного на этапе оценки модели.

1. Импортирует json и Path.
2. Читает файл reports/metrics.json.
3. Преобразует JSON в Python-словарь с помощью json.loads().
4. Выводит данные в удобном форматированном виде через json.dumps() с отступами и сохранением неанглийских символов.

В отчёте содержатся размер тестовой выборки, количество признаков, значение baseline и метрики модели и baseline: MAE, RMSE и R².

**Интерпретация**

В первом прогоне модель показала MAE около 2087.53, RMSE около 8128.65 и R² около 0.0012. У baseline соответствующие значения составили примерно 2216.37, 8224.00 и -0.0224.

Таким образом, модель лучше baseline по MAE и RMSE, однако R² близок к нулю. Это указывает на очень ограниченную объясняющую способность модели на тестовой выборке.

Сохранение метрик в машиночитаемом формате позволяет автоматически сравнивать качество моделей и отслеживать изменения между версиями данных. В дальнейшем эти значения можно использовать для анализа экспериментов и контроля качества.

In [ ]:
import json
from pathlib import Path

m = json.loads(Path("reports/metrics.json").read_text())
print(json.dumps(m, indent=2, ensure_ascii=False))

{
  "test_size": 7951,
  "n_features": 59,
  "baseline_mean_shares": 1803.435730154749,
  "model_mae": 2087.5340787465866,
  "model_rmse": 8128.64778387328,
  "model_r2": 0.0011814479746963258,
  "baseline_mae": 2216.3654265198484,
  "baseline_rmse": 8224.004241995659,
  "baseline_r2": -0.022390110377184635
}


## 21. pre-commit: установка хуков и прогон по всем файлам

Устанавливает Git-хуки pre-commit и запускает настроенные проверки для всех файлов проекта.

1. Первый вызов pre-commit run --all-files устанавливает необходимые окружения хуков и запускает проверки.
2. В частности, выполняются:
   - ruff-check — поиск и автоматическое исправление ошибок Python-кода;
   - ruff-format — форматирование Python-файлов;
   - check-added-large-files — проверка слишком больших файлов;
   - check-yaml — проверка YAML-файлов;
   - detect-private-key — поиск случайно добавленных приватных ключей.
3. Второй запуск повторяет проверки после возможных автоматических исправлений.
4. По коду завершения определяется, прошли ли проверки успешно.

Pre-commit автоматизирует контроль качества кода до коммита. Это помогает поддерживать единый стиль проекта, обнаруживать типовые ошибки и снижать риск попадания секретов или крупных файлов в репозиторий.

In [ ]:
import subprocess

print("Прогон 1 (авто-исправление)")
r = subprocess.run(["pre-commit", "run", "--all-files"], capture_output=True, text=True)
print(r.stdout[-4000:] if r.stdout else "")
print(r.stderr[-1000:] if r.stderr else "")

print("Прогон 2 (проверка)")
r2 = subprocess.run(["pre-commit", "run", "--all-files"], capture_output=True, text=True)
print(r2.stdout[-4000:] if r2.stdout else "")
if r2.returncode == 0:
    print("pre-commit: всё чисто.")
else:
    print("pre-commit нашёл проблемы — исправь код перед коммитом.")

Прогон 1 (авто-исправление)
[INFO] Initializing environment for https://github.com/astral-sh/ruff-pre-commit.
[INFO] Initializing environment for https://github.com/pre-commit/pre-commit-hooks.
[INFO] Installing environment for https://github.com/astral-sh/ruff-pre-commit.
[INFO] Once installed this environment will be reused.
[INFO] This may take a few minutes...
[INFO] Installing environment for https://github.com/pre-commit/pre-commit-hooks.
[INFO] Once installed this environment will be reused.
[INFO] This may take a few minutes...
ruff check...........................................(no files to check)Skipped
ruff format..........................................(no files to check)Skipped
check for added large files..............................................Passed
check yaml...........................................(no files to check)Skipped
detect private key.......................................................Passed


Прогон 2 (проверка)
ruff check..........................

## 22. README

Создаёт README-файл с описанием проекта, инструкциями по воспроизведению эксперимента и документацией по структуре ML-пайплайна.


In [ ]:
from pathlib import Path

Path("README.md").write_text(
    "# Online News Popularity — воспроизводимый ML-пайплайн\n"
    "\n"
    "[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?logo=python&logoColor=white)](https://www.python.org/)\n"
    "[![DVC](https://img.shields.io/badge/data%20versioning-DVC-945DD6)](https://dvc.org/)\n"
    "[![Task](https://img.shields.io/badge/task-regression-2E8B57)](#постановка-задачи)\n"
    "\n"
    "Репозиторий лабораторной работы по MLOps: версионирование данных с DVC, воспроизводимый пайплайн обработки и обучения, фиксация метрик и демонстрация перехода между версиями датасета.\n"
    "\n"
    "## Содержание\n"
    "\n"
    "- [Постановка задачи](#постановка-задачи)\n"
    "- [Окружение и установка](#окружение-и-установка)\n"
    "- [Получение данных](#получение-данных)\n"
    "- [Запуск пайплайна](#запуск-пайплайна)\n"
    "- [Граф пайплайна](#граф-пайплайна)\n"
    "- [Результаты и метрики](#результаты-и-метрики)\n"
    "- [Смена версии данных](#смена-версии-данных)\n"
    "- [Проверки качества](#проверки-качества)\n"
    "- [Структура проекта](#структура-проекта)\n"
    "\n"
    "## Постановка задачи\n"
    "\n"
    "**Вариант 7 — Online News Popularity (UCI).** Требуется решить задачу регрессии: предсказать количество репостов статьи (`shares`) по её признакам.\n"
    "\n"
    "Разбиение выполняется по времени с использованием `timedelta`: старые статьи используются для обучения, более новые — для тестирования, имитирующего поток новых публикаций. В этом наборе большее значение `timedelta` соответствует более старой статье; поэтому в обучающую выборку попадают строки с большими значениями, а в тестовую — с меньшими значениями относительно выбранной границы. Такая схема помогает избежать временной утечки.\n"
    "\n"
    "Модель обучается на `log1p(shares)`, а предсказания преобразуются обратно через `expm1` и ограничиваются снизу нулём. Модель по умолчанию — `HistGradientBoostingRegressor` (`train.model: hist_gb`). В качестве ориентира рассчитывается baseline, предсказывающий константу на основе среднего `log1p(shares)` обучающей выборки.\n"
    "\n"
    "## Окружение и установка\n"
    "\n"
    "Требуются Python 3.10 или новее, Git, `uv` и доступ к репозиторию GitHub и DVC-хранилищу в DagsHub.\n"
    "\n"
    "```bash\n"
    "git clone https://github.com/IsamuYosida/lr1newspopularity.git\n"
    "cd lr1newspopularity\n"
    "\n"
    "# Установить менеджер окружения\n"
    "python -m pip install uv\n"
    "\n"
    "# Установить зависимости из зафиксированного lock-файла\n"
    "uv sync --frozen\n"
    "```\n"
    "\n"
    "Если проект запускается не через `uv`, можно экспортировать зафиксированные зависимости и установить их через `pip`:\n"
    "\n"
    "```bash\n"
    "uv export --frozen --no-hashes -o requirements.lock.txt\n"
    "python -m pip install -r requirements.lock.txt\n"
    "```\n"
    "\n"
    "## Получение данных\n"
    "\n"
    "Исходный CSV и артефакты хранятся в DVC remote на DagsHub, а в Git находятся `.dvc`-метафайлы, связывающие файлы с конкретными версиями. Для загрузки нужен DagsHub Personal Access Token.\n"
    "\n"
    "Настройте доступ к DVC-хранилищу локально:\n"
    "\n"
    "```bash\n"
    "dvc remote modify --local origin access_key_id <DAGSHUB_TOKEN>\n"
    "dvc remote modify --local origin secret_access_key <DAGSHUB_TOKEN>\n"
    "```\n"
    "\n"
    "Замените `<DAGSHUB_TOKEN>` своим токеном. Не вставляйте токен в README, код или Git. Команды с `--local` сохраняют настройки в `.dvc/config.local`, который не должен попадать в коммит.\n"
    "\n"
    "Загрузите данные и доступные DVC-артефакты:\n"
    "\n"
    "```bash\n"
    "dvc pull\n"
    "```\n"
    "\n"
    "Если при `dvc pull` DVC сообщает, что часть выходных файлов отсутствует в remote, это не мешает пересоздать их через пайплайн, если исходный CSV доступен.\n"
    "\n"
    "## Запуск пайплайна\n"
    "\n"
    "Из корня репозитория выполните:\n"
    "\n"
    "```bash\n"
    "uv run dvc repro\n"
    "```\n"
    "\n"
    "Либо, если зависимости установлены в активное окружение:\n"
    "\n"
    "```bash\n"
    "dvc repro\n"
    "```\n"
    "\n"
    "DVC выполнит только те стадии, у которых изменились входы, код или параметры. Для принудительного повторного выполнения всех стадий:\n"
    "\n"
    "```bash\n"
    "dvc repro --force\n"
    "```\n"
    "\n"
    "Посмотреть метрики:\n"
    "\n"
    "```bash\n"
    "cat reports/metrics.json\n"
    "```\n"
    "\n"
    "Посмотреть граф зависимостей:\n"
    "\n"
    "```bash\n"
    "dvc dag\n"
    "```\n"
    "\n"
    "## Граф пайплайна\n"
    "\n"
    "```text\n"
    "data/external/OnlineNewsPopularity.csv\n"
    "                  |\n"
    "                  v\n"
    "              prepare\n"
    "                  |\n"
    "                  v\n"
    "              featurize\n"
    "                  |\n"
    "                  v\n"
    "                train\n"
    "                  |\n"
    "                  v\n"
    "               evaluate\n"
    "                  |\n"
    "                  v\n"
    "         reports/metrics.json\n"
    "```\n"
    "\n"
    "| Стадия | Назначение | Основные результаты |\n"
    "|---|---|---|\n"
    "| `prepare` | Проверка схемы и временное разбиение данных | `data/processed/train.csv`, `test.csv` |\n"
    "| `featurize` | Подготовка матриц признаков и целевой переменной | `data/features/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` |\n"
    "| `train` | Обучение модели на обучающей выборке | `models/model.joblib` |\n"
    "| `evaluate` | Расчёт метрик модели и baseline, проверка порога `min_r2` | `reports/metrics.json` |\n"
    "\n"
    "Определение стадий и их зависимостей находится в `dvc.yaml`, параметры — в `params.yaml`, зафиксированные версии зависимостей стадий — в `dvc.lock`.\n"
    "\n"
    "## Результаты и метрики\n"
    "\n"
    "Ниже приведены результаты сохранённых прогонов для двух версий данных. Метрики считаются на тестовой выборке, состоящей из более новых статей. MAE и RMSE измеряются в количестве репостов; для них меньшие значения лучше. Для `R²` большее значение лучше.\n"
    "\n"
    "### Метрики модели\n"
    "\n"
    "| Версия данных | Всего строк | Строк в тесте | MAE | RMSE | R² |\n"
    "|---|---:|---:|---:|---:|---:|\n"
    "| v1 (`data-v1`) | 39 644 | 7 951 | 2087.53 | 8128.65 | 0.0012 |\n"
    "| v2 (`data-v2`) | 40 644 | 8 106 | 2081.58 | 8089.39 | 0.0012 |\n"
    "\n"
    "### Baseline\n"
    "\n"
    "| Версия данных | MAE | RMSE | R² |\n"
    "|---|---:|---:|---:|\n"
    "| v1 (`data-v1`) | 2216.37 | 8224.00 | -0.0224 |\n"
    "| v2 (`data-v2`) | 2215.69 | 8186.54 | -0.0230 |\n"
    "\n"
    "Модель немного превосходит baseline по MAE и RMSE, однако `R²` близок к нулю. Это означает, что в текущей конфигурации признаки слабо объясняют разброс количества репостов. Числа выше — результаты сохранённых прогонов; при изменении кода, параметров или состава данных метрики могут измениться.\n"
    "\n"
    "Полный набор скалярных значений, включая число признаков и метрики baseline, записывается в `reports/metrics.json`. Для сравнения метрик двух Git-ревизий можно использовать:\n"
    "\n"
    "```bash\n"
    "dvc metrics diff data-v1 data-v2\n"
    "```\n"
    "\n"
    "## Смена версии данных\n"
    "\n"
    "Версии данных связаны с Git-тегами и соответствующими `.dvc`-метафайлами. DVC хранит содержимое данных в своём кэше/remote, а Git фиксирует метаданные и состояние кода.\n"
    "\n"
    "- **`data-v1`** — исходный датасет UCI, 39 644 строки.\n"
    "- **`data-v2`** — к исходному набору добавлены 1 000 строк, полученных копированием строк с наибольшим `timedelta` и присвоением им новых значений `timedelta = max_td + 1 + i`. Размер версии — 40 644 строки.\n"
    "\n"
    "Это искусственное изменение предназначено для демонстрации воспроизводимого переключения версий и пересчёта метрик; оно не является реальным поступлением новых статей. При воспроизведении следует учитывать, что добавленные строки — копии существующих наблюдений с изменённым временным признаком.\n"
    "\n"
    "### Переключение между версиями\n"
    "\n"
    "Сначала убедитесь, что рабочее дерево не содержит незакоммиченных изменений. Затем переключитесь на нужный Git-тег и синхронизируйте файлы данных:\n"
    "\n"
    "```bash\n"
    "# Перейти к исходной версии\n"
    "git checkout data-v1\n"
    "dvc checkout\n"
    "dvc repro\n"
    "cat reports/metrics.json\n"
    "\n"
    "# Перейти к версии v2\n"
    "git checkout data-v2\n"
    "dvc checkout\n"
    "dvc repro\n"
    "cat reports/metrics.json\n"
    "```\n"
    "\n"
    "Чтобы вернуться к основной ветке:\n"
    "\n"
    "```bash\n"
    "git checkout main\n"
    "dvc checkout\n"
    "dvc repro\n"
    "```\n"
    "\n"
    "`git checkout` переключает код и DVC-метафайлы, а `dvc checkout` восстанавливает соответствующие версии файлов данных из локального DVC-кэша. Если нужных объектов нет локально, сначала выполните `dvc pull` при настроенном доступе к remote. Не пропускайте `dvc checkout`: иначе в рабочей папке могут остаться данные от другой версии.\n"
    "\n"
    "## Проверки качества\n"
    "\n"
    "```bash\n"
    "# Установить Git-хуки (один раз)\n"
    "uv run pre-commit install\n"
    "\n"
    "# Запустить проверки по всем файлам\n"
    "uv run pre-commit run --all-files\n"
    "\n"
    "# Запустить smoke-тесты\n"
    "uv run pytest -q\n"
    "```\n"
    "\n"
    "В проекте используются проверки схемы данных, согласованности признаков обучающей и тестовой выборок, отсутствия пропусков/нечисловых признаков и порога `evaluate.min_r2`.\n"
    "\n"
    "## Структура проекта\n"
    "\n"
    "```text\n"
    ".\n"
    "├── src/\n"
    "│   ├── schema.py       # схема и валидация исходных данных\n"
    "│   ├── prepare.py      # валидация и временное разбиение\n"
    "│   ├── featurize.py    # подготовка X и y\n"
    "│   ├── train.py        # обучение модели\n"
    "│   └── evaluate.py     # метрики модели и baseline\n"
    "├── tests/\n"
    "│   └── test_smoke.py   # smoke-тесты\n"
    "├── data/\n"
    "│   ├── external/       # исходный CSV и .dvc-метафайл\n"
    "│   ├── processed/      # train/test после разбиения\n"
    "│   └── features/       # подготовленные признаки и target\n"
    "├── models/             # обученная модель\n"
    "├── reports/\n"
    "│   └── metrics.json    # итоговые метрики\n"
    "├── params.yaml         # параметры стадий\n"
    "├── dvc.yaml            # описание стадий пайплайна\n"
    "├── dvc.lock            # фиксация входов/выходов стадий\n"
    "├── pyproject.toml      # зависимости и конфигурация инструментов\n"
    "├── uv.lock             # зафиксированные версии зависимостей\n"
    "└── README.md\n"
    "```\n"
    "\n"
    "## Что хранится в Git и DVC\n"
    "\n"
    "- **Git:** исходный код, тесты, конфигурация, `uv.lock`, `dvc.yaml`, `dvc.lock`, `.dvc`-метафайлы и README.\n"
    "- **DVC remote:** исходный CSV и большие генерируемые артефакты, если они добавлены в DVC.\n"
    "- **Локально генерируются:** промежуточные данные, модель и отчёт при выполнении пайплайна. `reports/metrics.json` также может фиксироваться в Git для удобного сравнения результатов.\n"
    "- **Секреты:** токен DagsHub хранится только в локальной конфигурации и не коммитится.\n",
    encoding="utf-8",
)
print("README.md записан.")

README.md записан.


## 23. Коммит v1

Фиксирует первую версию проекта и данных в Git и DVC, создаёт коммит и устанавливает тег data-v1.

1. Переходит в ветку main с помощью git checkout -B main.
2. Выполняет dvc push, чтобы отправить текущие данные и артефакты в удалённое хранилище.
3. Добавляет файлы в Git через git add -A, отдельно добавляет reports/metrics.json и .dvc/config.
4. Удаляет из Git-индекса src/__pycache__, чтобы не сохранять служебные файлы Python.
5. Временно отключает существующий pre-commit hook, если он установлен, чтобы он не блокировал создание первого коммита. После операции hook возвращается на место через finally.
6. Создаёт коммит с сообщением LR1: data v1, pipeline, dvc.yaml, README, tests.
7. Проверяет, что HEAD указывает на существующий коммит.
8. Создаёт или обновляет Git-тег data-v1.
9. Отправляет изменения и тег на GitHub через функцию git_push().

Тег data-v1 фиксирует состояние проекта и метафайлов DVC для первой версии данных. Благодаря этому можно вернуться к определённому состоянию кода и восстановить соответствующий набор данных, а затем повторить эксперимент.

In [ ]:
import subprocess, os
from pathlib import Path
os.chdir(REPO_DIR)

def sh(*args, check=False, cwd=None):
    r = subprocess.run(args, capture_output=True, text=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(
            f"{args} -> rc={r.returncode}\nstdout={r.stdout}\nstderr={r.stderr}"
        )
    return r

# 0. Ветка main (создаст ref даже в пустом клоне)
sh("git", "checkout", "-B", "main", cwd=REPO_DIR)
# 1. dvc push — данные в DagsHub
print("1. dvc push...")
sh("dvc", "push", check=True, cwd=REPO_DIR)
# 2. Индекс
print("2. git add...")
sh("git", "add", "-A", cwd=REPO_DIR)
sh("git", "add", "-f", "reports/metrics.json", cwd=REPO_DIR)
sh("git", "add", ".dvc/config", cwd=REPO_DIR)
sh("git", "rm", "-r", "--cached", "src/__pycache__", cwd=REPO_DIR)
# 2b. Если pre-commit hook остался с прошлого прогона — отключаем
hook = Path(".git/hooks/pre-commit")
hook_backup = Path(".git/hooks/pre-commit.disabled")
hook_was_there = hook.exists()
if hook_was_there:
    hook.rename(hook_backup)
    print("pre-commit hook временно отключён для первого коммита.")

try:
    # 3. Коммит
    print("3. git commit...")
    r = sh("git", "commit", "-m",
           "LR1: data v1, pipeline, dvc.yaml, README, tests", cwd=REPO_DIR)
    if r.returncode == 0:
        print("Коммит создан.")
    elif "nothing to commit" in (r.stdout + r.stderr):
        print("Уже есть коммит.")
    else:
        raise RuntimeError(f"git commit упал:\nstdout={r.stdout}\nstderr={r.stderr}")
finally:
    if hook_was_there and hook_backup.exists():
        hook_backup.rename(hook)
        print("pre-commit hook возвращён на место.")

# 3b. HEAD валиден?
head = sh("git", "rev-parse", "--verify", "HEAD", cwd=REPO_DIR)
if head.returncode != 0:
    raise RuntimeError("HEAD всё ещё невалиден — коммит не создан.")
print("HEAD:", head.stdout.strip()[:12])

# 4. Тег data-v1
print("4. git tag -f data-v1...")
r = sh("git", "tag", "-f", "data-v1", cwd=REPO_DIR)
if r.returncode != 0:
    raise RuntimeError(f"git tag упал: {r.stderr}")
print("Тег data-v1 поставлен.")

# 5. Push на GitHub
print("5. git push...")
git_push("(тег data-v1)")

1. dvc push...
2. git add...
3. git commit...
Коммит создан.
HEAD: cc0374edffdd
4. git tag -f data-v1...
Тег data-v1 поставлен.
5. git push...
git push: OK (тег data-v1)


После создания первого коммита устанавливает Git-хуки pre-commit и повторно проверяет все файлы репозитория.

1. Переходит в каталог репозитория REPO_DIR.
2. Выполняет pre-commit install, чтобы подключить хуки к Git.
3. Запускает pre-commit run --all-files.
4. Объединяет стандартный вывод и сообщения об ошибках, после чего печатает последние 3000 символов.
5. По коду завершения сообщает, прошли ли проверки.

В сохранённом выводе Ruff обнаружил две ошибки и автоматически исправил их. Форматтер также изменил шесть файлов. Поэтому первый запуск отмечен как неуспешный: хуки внесли изменения. После этого проверки следует повторить.


In [ ]:
import subprocess, os
os.chdir(REPO_DIR)

# Первый коммит уже прошёл — теперь можно ставить хуки,
# они не заблокируют коммит v1.
print("1. pre-commit install...")
subprocess.run(["pre-commit", "install"], check=True, cwd=REPO_DIR)
print("Хуки установлены.")

print("2. pre-commit run --all-files...")
r = subprocess.run(
    ["pre-commit", "run", "--all-files"],
    capture_output=True, text=True, cwd=REPO_DIR,
)
out = (r.stdout or "") + (r.stderr or "")
print(out[-3000:] if out else "")

if r.returncode == 0:
    print("pre-commit: чисто.")
else:
    print("pre-commit: есть замечания.")

1. pre-commit install...
Хуки установлены.
2. pre-commit run --all-files...
ruff check...............................................................Failed
- hook id: ruff-check
- files were modified by this hook

Found 2 errors (2 fixed, 0 remaining).

ruff format..............................................................Failed
- hook id: ruff-format
- files were modified by this hook

6 files reformatted, 1 file left unchanged

check for added large files..............................................Passed
check yaml...............................................................Passed
detect private key.......................................................Passed

pre-commit: есть замечания.


## 24. Версия данных v2

Создаёт вторую версию датасета, увеличивая исходный набор на 1000 строк, после чего обновляет DVC-версию данных и повторно запускает ML-пайплайн.

1. Загружает исходный CSV и проверяет, что в нём нет отрицательных значений timedelta.
2. Сохраняет исходное количество строк в n_v1.
3. Выбирает 1000 строк с наибольшим timedelta через nlargest(1000, "timedelta").
4. Создаёт копию выбранных строк и изменяет их timedelta: новым значениям присваивается последовательность после максимального значения исходного набора.
5. Объединяет исходные данные и добавленные строки через pd.concat(), получая версию v2.
6. Сохраняет обновлённый CSV.
7. Выполняет dvc add -f и dvc push, чтобы обновить метафайл DVC и отправить новую версию данных в удалённое хранилище.
8. Запускает dvc repro, пересчитывая необходимые стадии пайплайна.
9. Читает новые метрики из reports/metrics.json.
10. Создаёт Git-коммит, устанавливает тег data-v2 и отправляет изменения на GitHub.

В этой реализации новые строки получаются копированием 1000 существующих записей, а их `timedelta` искусственно увеличивается. Это демонстрационный сценарий управления версиями данных, а не получение 1000 новых реальных новостных статей.



In [ ]:
import pandas as pd, subprocess, json, os
from pathlib import Path
os.chdir(REPO_DIR)

def sh(*args, check=False, cwd=None):
    r = subprocess.run(args, capture_output=True, text=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f"{args} -> rc={r.returncode}\nstdout={r.stdout}\nstderr={r.stderr}")
    return r

csv_path = Path("data/external/OnlineNewsPopularity.csv")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
assert (df["timedelta"] < 0).sum() == 0, "Ошибка: в данных есть отрицательные timedelta!"
n_v1 = len(df)

fresh = df.nlargest(1000, "timedelta").copy()
max_td = float(df["timedelta"].max())
fresh["timedelta"] = max_td + 1.0 + pd.RangeIndex(len(fresh)).astype(float)

df_v2 = pd.concat([df, fresh], ignore_index=True)
df_v2.to_csv(csv_path, index=False)
print(f"v2: {len(df_v2)} строк (было {n_v1}), добавлено {len(fresh)}")
print(f"min timedelta: {df_v2['timedelta'].min()}  max timedelta: {df_v2['timedelta'].max()}")

print("\n1. dvc add -f + push...")
sh("dvc", "add", "-f", "data/external/OnlineNewsPopularity.csv", check=True, cwd=REPO_DIR)
sh("dvc", "push", check=True, cwd=REPO_DIR)

print("\n2. dvc repro...")
sh("dvc", "repro", check=True, cwd=REPO_DIR)

m_v2 = json.loads(Path("reports/metrics.json").read_text())
print("\nМетрики v2:", m_v2)

print("\n3. git add + commit...")
sh("git", "add", "-A", cwd=REPO_DIR)
sh("git", "add", "-f", "reports/metrics.json", cwd=REPO_DIR)
sh("git", "rm", "-r", "--cached", "src/__pycache__", cwd=REPO_DIR)
r = sh("git", "commit", "-m", "LR1: data v2 (new stream)", cwd=REPO_DIR)
if r.returncode == 0:
    print("   Коммит создан.")
elif "nothing to commit" in (r.stdout + r.stderr):
    print("   Нечего коммитить.")
else:
    raise RuntimeError(f"git commit упал:\n{r.stderr}")

# HEAD должен быть валиден после коммита v1
head = sh("git", "rev-parse", "--verify", "HEAD", cwd=REPO_DIR)
if head.returncode != 0:
    raise RuntimeError("HEAD невалиден перед тегом data-v2.")

print("\n4. git tag -f data-v2...")
r = sh("git", "tag", "-f", "data-v2", cwd=REPO_DIR)
if r.returncode != 0:
    raise RuntimeError(f"git tag data-v2 упал: {r.stderr}")
print("   Тег data-v2 поставлен.")

print("\n5. git push...")
git_push("(тег data-v2)")

v2: 40644 строк (было 39644), добавлено 1000
min timedelta: 8.0  max timedelta: 1731.0

1. dvc add -f + push...

2. dvc repro...

Метрики v2: {'test_size': 8106, 'n_features': 59, 'baseline_mean_shares': 1796.6765382856502, 'model_mae': 2081.581046054519, 'model_rmse': 8089.392225579732, 'model_r2': 0.0011835667622626111, 'baseline_mae': 2215.6934711667363, 'baseline_rmse': 8186.542004025022, 'baseline_r2': -0.02295111814246731}

3. git add + commit...
   Коммит создан.

4. git tag -f data-v2...
   Тег data-v2 поставлен.

5. git push...
git push: OK (тег data-v2)


##25. Откат к v1 через кэш

Демонстрирует восстановление первой версии датасета и повторное выполнение пайплайна на v1, после чего возвращает проект к версии v2.

1. Переходит в каталог репозитория и определяет путь к исходному CSV.
2. Функция restore_from_cache() читает DVC-метафайл OnlineNewsPopularity.csv.dvc` и извлекает из него MD5-хеш файла.
3. По хешу формируется путь к объекту в локальном DVC-кэше: .dvc/cache/files/md5.
4. Если файл найден, он копируется обратно в data/external/OnlineNewsPopularity.csv. Если его нет, вызывается ошибка.
5. Выполняется git checkout -f data-v1, после чего восстанавливается соответствующий CSV из DVC-кэша.
6. Проверяется количество строк, затем запускается dvc repro для версии v1.
7. Из отчёта извлекаются основные метрики модели.
8. В конце выполняется переход обратно на main, а CSV восстанавливается из кэша для текущей версии v2.

Ячейка показывает совместную работу Git и DVC при переключении между версиями проекта и данных. Git восстанавливает код и метафайлы, а DVC позволяет вернуть соответствующие версии больших файлов.

In [ ]:
import subprocess, json, os, yaml, shutil
from pathlib import Path
os.chdir(REPO_DIR)

def sh(*args, check=False, cwd=None):
    r = subprocess.run(args, capture_output=True, text=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f"{args} -> rc={r.returncode}\n{r.stderr}")
    return r

csv = Path("data/external/OnlineNewsPopularity.csv")

def restore_from_cache():
    """Восстанавливает CSV из локального DVC-кэша по md5 из .dvc-файла."""
    meta = yaml.safe_load(Path("data/external/OnlineNewsPopularity.csv.dvc").read_text())
    md5 = meta["outs"][0]["md5"]
    cache_file = Path(".dvc/cache/files/md5") / md5[:2] / md5[2:]
    if not cache_file.exists():
        raise RuntimeError(f"Кэш-файл {cache_file} отсутствует — данные потеряны.")
    shutil.copy(cache_file, csv)
    print(f"CSV восстановлен из кэша ({md5[:8]}...)")

print("1. git checkout -f data-v1...")
sh("git", "checkout", "-f", "data-v1", check=True, cwd=REPO_DIR)

print("2. Восстанавливаем CSV из кэша по md5...")
restore_from_cache()

df_back = pd.read_csv(csv)
print(f"Строк после отката: {len(df_back)}  (ожидается 39644)")

print("3. dvc repro на v1...")
sh("dvc", "repro", check=True, cwd=REPO_DIR)
m_v1 = json.loads(Path("reports/metrics.json").read_text())

print("Метрики v1:", {k: m_v1[k] for k in ("model_mae", "model_rmse", "model_r2")})
print("4. Возврат на main...")
sh("git", "checkout", "-f", "main", check=True, cwd=REPO_DIR)
restore_from_cache()
print("Вернулись на main, данные v2 восстановлены.")

1. git checkout -f data-v1...
2. Восстанавливаем CSV из кэша по md5...
CSV восстановлен из кэша (6e6f1cfc...)
Строк после отката: 39644  (ожидается 39644)
3. dvc repro на v1...
Метрики v1: {'model_mae': 2087.5340787465866, 'model_rmse': 8128.64778387328, 'model_r2': 0.0011814479746963258}
4. Возврат на main...
CSV восстановлен из кэша (88cdb5d5...)
Вернулись на main, данные v2 восстановлены.


##26. Сохранение ноутбука и финальный коммит

Выполняет финальную синхронизацию данных и артефактов с DagsHub, фиксирует оставшиеся изменения в Git и отправляет их на GitHub.

1. Переходит в каталог проекта REPO_DIR.
2. Выполняет dvc push, чтобы отправить актуальные данные и артефакты в удалённое хранилище.
3. Добавляет изменения в Git с помощью git add -A.
4. Отдельно добавляет reports/metrics.json, поскольку каталог отчётов исключён из обычного отслеживания.
5. Удаляет из Git-индекса src/__pycache__.
6. Пытается создать финальный коммит с сообщением LR1: final. Если изменений нет, выводит соответствующее сообщение вместо ошибки.
7. Выполняет git push() для отправки изменений на удалённый репозиторий.



In [ ]:
import subprocess, os
from pathlib import Path
os.chdir(REPO_DIR)

def sh(*args, check=False, cwd=None):
    r = subprocess.run(args, capture_output=True, text=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f"{args} -> rc={r.returncode}\n{r.stderr}")
    return r

print("1. dvc push...")
sh("dvc", "push", check=True, cwd=REPO_DIR)

print("2. Финальный коммит...")
sh("git", "add", "-A", cwd=REPO_DIR)
sh("git", "add", "-f", "reports/metrics.json", cwd=REPO_DIR)
sh("git", "rm", "-r", "--cached", "src/__pycache__", cwd=REPO_DIR)
r = sh("git", "commit", "-m", "LR1: final", cwd=REPO_DIR)
if r.returncode == 0:
    print("Коммит создан.")
elif "nothing to commit" in (r.stdout + r.stderr):
    print("Нечего коммитить.")
else:
    raise RuntimeError(f"git commit упал:\n{r.stderr}")

print("3. git push...")
git_push()
print("\nГотово!")

1. dvc push...
2. Финальный коммит...
Нечего коммитить.
3. git push...
git push: OK 

Готово!


#Контрольные вопросы

1. Что фиксирует uv.lock и почему его недостаточно для воспроизводимости ML-проекта?

uv.lock фиксирует версии Python-зависимостей и их разрешённые комбинации, чтобы окружение можно было восстановить одинаково. Но для воспроизводимости ML-проекта этого недостаточно: нужно также версионировать исходный код, данные, параметры эксперимента и этапы обработки. Именно поэтому в проекте дополнительно используются Git и DVC.

2. Что хранится в Git, что в хранилище DVC и как они связаны между собой?

- Git хранит исходный код, конфигурации, dvc.yaml, params.yaml и .dvc-файлы с метаданными данных.
- DVC-хранилище хранит сами большие файлы данных и артефакты.
- Связь обеспечивается метаданными DVC: они содержат контрольные суммы и указывают, какие версии файлов соответствуют конкретному коммиту Git.

3. Как DVC определяет, какие этапы пайплайна нужно пересчитать?

DVC анализирует зависимости и выходы этапов, описанные в dvc.yaml, а также параметры из params.yaml. Он сравнивает контрольные суммы файлов и конфигураций с предыдущим запуском. Если входы и параметры не изменились, этап можно пропустить; если изменились — пересчитать его и зависимые от него этапы.

4. Почему ключи доступа задаются с флагом --local, а адрес хранилища — без него?

Адрес хранилища — общая настройка проекта, поэтому он сохраняется в .dvc/config и может быть передан другим участникам через Git. Ключи доступа — конфиденциальные данные, поэтому их сохраняют локально в .dvc/config.local, который не должен попадать в репозиторий.

5. Почему dvc push выполняется раньше git push?

Сначала dvc push загружает данные и артефакты в удалённое DVC-хранилище. Затем git push публикует коммит, содержащий ссылки на эти версии данных. Такой порядок снижает риск ситуации, когда в Git уже появилась версия проекта, а соответствующих ей файлов в DVC-хранилище ещё нет.

6. Какие две команды возвращают проект к состоянию месячной давности вместе с данными?

В общем случае:

```
git checkout <нужный-коммит-или-тег>
dvc checkout
```

Первая команда возвращает код и конфигурации к выбранной версии Git, вторая восстанавливает соответствующие версии файлов данных из локального DVC-кэша. Если нужных файлов в кэше нет, предварительно потребуется dvc pull.

7. Почему случайное разбиение данных с датой завышает оценку качества?

При случайном разбиении в обучающую и тестовую выборки могут попасть статьи из одного и того же временного периода. Модель обучается на данных, которые по времени близки к тестовым, поэтому проверка хуже имитирует реальное применение — предсказание популярности будущих публикаций по прошлым данным. Это может привести к завышенной оценке качества.

В проекте используется временное разделение по timedelta: тестовая выборка содержит более новые статьи, а обучающая — более старые. Это позволяет оценивать модель ближе к реальному сценарию использования.